# ConvNeXt-Tiny classification — Oral cancer histopathology

Trains `train_classifier.py` from [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) on whole stain-normalized 512×512 patches (resized to 224).

MedSAM segmentation was evaluated separately: its masks covered ~95% of every patch, so ROIs were effectively the full patch. Classification therefore uses whole patches.

| Run | Purpose |
|---|---|
| **4-class** | Main result (OSCC / with dysplasia / without dysplasia / Normal) |
| **3-class** | NDB-UFES classes only, comparable with published work (e.g. Maia et al. 2024, DenseNet-121: 91.9% balanced accuracy) |
| **4-class greyscale** | Colour-shortcut check: if Normal stays easy without colour, the model uses tissue structure |

The best epoch is chosen on the **validation** split (macro-F1); the **test** split is evaluated once at the end.

### Before running
- **Add Input** → your private dataset with `splits/` and `preprocessed_normalized/` (same one as the segmentation notebook).
- **Settings** → Accelerator **GPU T4**, Internet **On**.
- Use **Save Version → Save & Run All** so all runs finish in the background (roughly 1–1.5 hours in total).

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)

print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

matches = list(Path("/kaggle/input").rglob("splits/normalized_dataset.csv"))
assert matches, "Could not find splits/normalized_dataset.csv under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent
OUTPUT_ROOT = Path("/kaggle/working")

os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

In [ ]:
import json
from IPython.display import Image as ShowImage, display

def show(run):
    d = OUTPUT_ROOT / "runs" / run
    display(ShowImage(str(d / "curves.png")))
    display(ShowImage(str(d / "confusion_matrix.png")))

## 2. Main run — 4 classes

In [ ]:
!cd {CODE_DIR} && python train_classifier.py --classes 4

In [ ]:
show("convnext_tiny_4class_224px")

## 3. NDB-UFES only — 3 classes (comparable with published results)

In [ ]:
!cd {CODE_DIR} && python train_classifier.py --classes 3

In [ ]:
show("convnext_tiny_3class_224px")

## 4. Colour-shortcut check — 4 classes in greyscale

Compare **Normal** recall/F1 with the colour run. If it stays high, the model recognises Normal tissue by structure; if it drops sharply, colour (i.e. the source dataset) was doing the work.

In [ ]:
!cd {CODE_DIR} && python train_classifier.py --classes 4 --grayscale

In [ ]:
show("convnext_tiny_4class_gray_224px")

## 5. Comparison

In [ ]:
import pandas as pd

rows = []
for metrics_file in sorted((OUTPUT_ROOT / "runs").glob("*/metrics.json")):
    m = json.load(open(metrics_file))
    t = m["test"]
    row = {
        "run": m["run"],
        "best epoch": m["best_epoch"],
        "accuracy": t["accuracy"],
        "balanced acc": t["balanced_accuracy"],
        "macro-F1": t["macro_f1"],
        "macro AUC": t["macro_auc_ovr"],
    }
    for cls, scores in t["per_class"].items():
        if isinstance(scores, dict) and cls not in ("macro avg", "weighted avg"):
            row[f"F1 {cls}"] = scores["f1-score"]
    if "ndb_ufes_subset" in t:
        row["NDB-only balanced acc"] = t["ndb_ufes_subset"]["balanced_accuracy"]
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("run").T.round(4)
comparison.to_csv(OUTPUT_ROOT / "runs" / "comparison.csv")
comparison

## 6. Package results for download

Creates `runs.zip` (models, metrics, predictions, plots) in the output panel. Unzip into the project folder on your laptop.

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_ROOT / "runs"), "zip", OUTPUT_ROOT, "runs")
print(archive, round(os.path.getsize(archive) / 2**20), "MB")